---
title: "The wt CLI source, walked through"
date: "2026-08-09"
categories: ["meta", "dev"]
toc: true
---

# `src/watchtower/`: package walkthrough

> Teaching notes for the `wt` CLI implementation. ~1,800 lines across 10 files.
> All content derived from reading the source; accurate as of Aug 2026.

## One-liner

The entire implementation of the `wt` CLI: a personal knowledge-base tool
whose primary file type is Jupyter notebooks (notes/articles/courses), plus
three peripheral subsystems (vault, resume, render) that share only the path
hub.

## Why read the source

The library was built fast, mostly through agent-assisted coding. A working
tool arrives quickly that way, but the result is 1,800 lines owned but not
internalized: the first break lands on code with no mental model behind it.
This page closes that gap.

The scale helps. 1,800 lines is small enough to read in full, and the
architecture is deliberately simple: no framework, no plugin system, no
stateful objects, no clever metaprogramming. What sits underneath is a set
of small, mostly stateless functions that funnel through a few choke
points. Once the shape is visible (one resolution function, one error
pattern, one I/O library, one path hub), the package reduces to a handful
of decisions rather than an undifferentiated mass.

The decisions are worth understanding on their own, because they are the
choices any small CLI tool faces: how to keep a CLI thin, where to enforce
invariants, how to make file I/O safe for an AI agent to drive, how to get
reproducible builds out of a tool that shells out to LaTeX. The `wt`
codebase is a worked example of each, small enough to hold at once.

## How to read this page

A documentation-style tour, in order:

1. [Module-by-module](#module-by-module): the map, one table of what each file does.
2. [Dependency graph](#dependency-graph): how the modules relate before reading any code.
3. [Modules in detail](#modules-in-detail): the per-file walkthrough with code snippets.
4. [One command, end to end](#one-command-end-to-end): trace a real `wt cat` through the layers.
5. [The test suite](#test-suite): what the 75 tests pin down, and what they deliberately skip.
6. [Core vs. peripheral](#core-vs-peripheral): what is load-bearing, what is swappable.
7. [Design patterns worth knowing](#design-patterns-worth-knowing): the ideas that recur across files, and why.
8. [Errors & constraints cheat-sheet](#errors-cheat-sheet): the rules that keep agents and the tool safe.

Sections 1-2 give the big picture; sections 3-5 are where the code gets
read (source then tests); the last three are reference material to come
back to.


## Module-by-module {#module-by-module}

Ten files, about 1,800 lines total. Start here for the map, then read the
dependency graph to see how the modules connect, then the details below.

| Module | Lines | Role |
|---|---|---|
| `paths.py` | 17 | Central path constants + repo-root detection, the dependency hub |
| `notebook.py` | 481 | Core read/write layer: all cell operations via nbformat |
| `inspect.py` | 144 | Read-only inspection: map, ls, find, name→path resolution |
| `scaffold.py` | 257 | Create notes/articles/courses/chapters/projects |
| `convert.py` | 96 | Import external notebooks (Colab/Kaggle), preserving outputs |
| `render.py` | 59 | Quarto subprocess wrappers (PDF render, site preview) |
| `resume.py` | 246 | Self-contained YAML → LaTeX PDF + site home page pipeline |
| `vault.py` | 47 | Secrets via OS keyring; fully isolated |
| `cli.py` | 458 | Typer app assembling every `wt` subcommand; thin dispatch |
| `__init__.py` | 2 | Package marker + `__version__ = "0.1.0"` |

Line counts are `wc -l` as of Aug 2026. The first four rows are the
notebook core; the last three (render, resume, vault) are peripheral
subsystems, discussed in [Core vs. peripheral](#core-vs-peripheral).


## Dependency graph {#dependency-graph}

Read the arrows as *imports*. The shape of this graph is the architecture:
nothing imports `cli.py`, so the CLI is a thin shell; nearly everything
eventually touches `paths.py`, so the filesystem layout has one source of
truth. The grouping into core layer and peripheral subsystems is discussed
in [Core vs. peripheral](#core-vs-peripheral).

```{mermaid}
flowchart TB
    cli["cli.py"]

    subgraph core [Core notebook layer]
        notebook["notebook.py"]
        inspect["inspect.py"]
        scaffold["scaffold.py"]
        convert["convert.py"]
    end

    subgraph peripheral [Peripheral subsystems]
        render["render.py"]
        resume["resume.py"]
        vault["vault.py"]
    end

    paths["paths.py"]

    cli --> notebook
    cli --> inspect
    cli --> convert
    cli --> scaffold
    cli --> render
    cli --> resume
    cli --> vault

    convert --> scaffold
    notebook --> inspect
    scaffold --> paths
    inspect --> paths
    resume --> paths

    classDef hub fill:#fff3bf,stroke:#e6a700,stroke-width:2px
    classDef peripheral fill:#f7f7f7,stroke:#999,stroke-dasharray:5 5
    class paths hub
    class render,resume,vault peripheral
```

- **Leaves (no internal deps):** `paths.py`, `vault.py`, `render.py`.
- **Strict layering:** arrows always point toward `paths.py` and the
  notebook primitives. No circular imports, so the modules are readable in
  dependency order.
- **`paths.py` is the hub:** every module that touches the filesystem
  imports it (directly or transitively). Change the repo layout in one
  place and the whole package follows.
- **Standalone subsystems:** `render.py` and `vault.py` have no internal
  imports at all. They are swappable and testable in isolation, which is
  why they sit at the edges of the graph.
- **Why the funnel shape:** all notebook traffic flows through
  `notebook.py` and `inspect.py`, so invariants (index-only writes, the
  resolution ladder) are enforced in exactly two places.


## Modules in detail {#modules-in-detail}

The first four modules form the notebook core: paths, the read/write layer,
and read-only inspection. The remaining five are covered in the next cell.

### `paths.py` (17 lines): the hub

- Constants: `NOTES_DIR`, `ARTICLES_DIR`, `COURSES_DIR`, `PROJECTS_DIR`, `CONTENT_DIRS`.
- `repo_root()` returns the repo root path.
- Stdlib only. Every module that touches the filesystem imports it
  (directly or transitively).

### `notebook.py` (481 lines): the core

All cell operations: reading notebooks as token-friendly markdown and
mutating through nbformat (never raw JSON).

- Constants: `MAX_CELL_SOURCE_CHARS = 20_000` (write cap),
  `DEFAULT_READ_LIMIT = 4096` (per-cell read slice).
- `count_cells(name)`, `cat_notebook(...)`: markdown rendering with
  `> cell N [kind] [tags:…] [label:…]` headers; char-sliceable source and
  per-output bodies; non-text outputs (images/base64, HTML) summarized.
- `edit_cell(...)`: replaces only `cells[i]["source"]`; outputs + metadata
  preserved. `append_cell(...)`, `insert_cell(...)`, `remove_cell(...)`
  (reverse-sorted deletion so earlier indices stay valid), `tag_cell(...)`.
- Locator machinery (the safeguard core):
  - `_parse_index_spec(spec, total)`: Python-slice-style `--index`
    (`N`, `N:M`, `:M`, `N:`, negatives), bounds-checked.
  - `_resolve_unique_cell(...)`: **writes require exactly one match**;
    errors on zero or >1 (ambiguity errors name the matched indices).
  - Tags/labels are read-only locators: usable with `wt cat` to *find*
    a cell, not accepted on writes.
- Imports: `nbformat`, `.inspect` (only `resolve_ipynb`).

### `inspect.py` (144 lines): read-only inspection

- `list_ipynb(src_dir)`: recursive listing, excluding `index.ipynb` and
  `.ipynb_checkpoints/`.
- `list_projects()`: project dirs containing `pyproject.toml`, with
  `name/path/has_agents_md`.
- `repo_map()` / `repo_map_json()`: the `wt map` JSON tree.
- `find_in_src(query)`: ripgrep pre-filter (`_candidate_ipynb_files`) then
  `json.load` parse only candidates; output `path [cell N]: line`.
- `resolve_ipynb(name)`: **the central name-resolution function**; bare
  stem / tier-prefixed (`notes/x`) / full path. Every notebook command
  funnels through this single choke point.
- Imports: `.paths`.


### `scaffold.py` (257 lines): content creation

- `new_note`, `new_article`: dated frontmatter stubs; title derived from name.
- `new_course`, `new_course_chapter`, `new_course_section`: creates notebooks
  + registers them in the course sidebar in `_quarto.yml`.
- `new_project(name)`: delegates to `uv init --package`.
- `_register_course` / `_register_chapter_in_sidebar`: idempotent sidebar
  surgery via **ruamel.yaml** (`typ="rt"`) so comments, key order, and
  quoting survive round-trips.
- Imports: `nbformat`, `ruamel.yaml`, `.paths`.

### `convert.py` (96 lines): external imports

- `import_notebook(src, tier, name=None)`: copy to `<tier>/<name>.ipynb`
  (flat tiers only; errors on `courses`; use `import_chapter` there).
- `import_chapter(src, course, chapter=None, section=None)`: copy into
  `courses/<course>/<chapter>.ipynb` + sidebar registration (reuses
  scaffold's `_register_chapter_in_sidebar`).
- `_copy_ipynb`: nbformat read/write, injecting a default Python-3
  kernelspec if missing (normalization step).
- Imports: `nbformat`, `.scaffold`.

### `render.py` (59 lines): Quarto wrappers

- `render_pdf(source)`: `quarto render <src> --to pdf` with
  `QUARTO_PYTHON=sys.executable` (the uv venv); finds the produced PDF
  (next to source or under `_site/`) and moves it into `notes/pdf/` or
  `articles/pdf/`.
- `preview_site()`: blocking `quarto preview`.
- Imports: stdlib only (subprocess/os/sys).

### `resume.py` (246 lines): standalone resume builder

- `build_resume() -> tuple[Path, Path]`: validates sources + `pdflatex`
  on PATH; loads YAML; Jinja2 env with three registered filters; renders
  `index.ipynb` (single markdown cell w/ Quarto frontmatter) and
  `resume.tex`; runs pdflatex **twice** in a stable scratch dir
  (`watchtower-resume-build`) under `SOURCE_DATE_EPOCH` = max source mtime.
  Result: **reproducible byte-identical builds** (deterministic PDF
  `/CreationDate` and `/ID`; pdfTeX hashes the build path into `/ID`).
- `_latex_text`: LaTeX escaping; markdown links→`\href`, bare URLs→`\url{}`.
- `_md_escape`: pandoc/markdown escaping for the web version; URLs raw so
  they autolink.
- `_html_entities`: numeric-entity obfuscation for the email (anti-scraper).
- `_escape_for_target(data, esc)`: deep-copies YAML dict, routes each
  free-text field through the right escaper; contact fields raw for
  `moderncv` macros.
- Imports: `nbformat`, `yaml`, `jinja2`, `.paths` (only `repo_root`).

### `vault.py` (47 lines): secrets, fully isolated

- OS keyring (service `"watchtower"`); values never touch disk, only a
  gitignored key index at `.watchtower/vault_keys.json`.
- `set_secret`, `get_secret`: wrap `keyring.set_password/get_password`.
- `list_keys`, `all_secrets`: backed by the JSON key index.
- Imports: `keyring`, stdlib only.

### `cli.py` (458 lines): the assembly

- `app: typer.Typer`: root CLI, `no_args_is_help=True`; `rich.Console` output.
- `new_app` / `vault_app`: nested Typer groups mounted via `add_typer`.
- `_user_error()` (contextmanager): catches `FileNotFoundError |
  FileExistsError | ValueError`, prints red, `typer.Exit(1)`. The package-wide
  error pattern.
- `_force_utf8_streams()` / `_read_stdin()`: pin stdio to UTF-8; reads
  piped content from `sys.stdin.buffer` (mojibake protection).
- `_open(path)`: platform opener for the rendered PDF.
- Every command is a thin wrapper around a function in another module.
- `pyproject.toml` maps `wt` and `watchtower` console scripts to
  `watchtower.cli:app`; `sys.exit(app())` under `__main__`.


## One command, end to end {#one-command-end-to-end}

Abstract layering is one thing; watching a real command cross every layer
makes it concrete. Trace `wt cat notes/005-wt-src-walkthrough --index 0`:

1. **`cli.py` parses the invocation.** The `cat` command (cli.py:214)
   resolves the read limit, defaulting to `notebook.DEFAULT_READ_LIMIT`
   (4096 chars) with a comment stating the reason: it protects agent
   context windows. The token-frugality design in action.
2. **`cli.py` delegates inside `_user_error()`.** The whole body runs in
   the context manager (cli.py:54), so any failure below is caught here,
   printed in red, and exits with status 1. No domain module ever prints
   or exits on its own.
3. **`cat_notebook` resolves the name.** `"notes/005-wt-src-walkthrough"`
   goes to `inspect.resolve_ipynb()`, the single choke point. It matches
   the tier-prefixed form: split on `/`, check the `notes` prefix, return
   `notes/005-wt-src-walkthrough.ipynb`.
4. **nbformat reads the file** (never raw JSON), and the matched cell
   (index 0) is rendered to markdown with its `> cell 0 [markdown]` header,
   then char-sliced to the limit.
5. **`cli.py` prints the result** to stdout with no trailing newline.

The same funnel applies to every command: resolve → operate → report.
Mutations add one step: after the operation, `nbformat.write` writes back,
with `edit_cell` touching only `cells[i]["source"]` so inline outputs
survive. The whole architecture in five steps: a thin CLI, a resolution
choke point, nbformat as the only I/O path, and one error pattern.


## The test suite {#test-suite}

The `tests/` directory mirrors `src/watchtower/` one file per module, plus
a shared `conftest.py`. 75 tests, ~680 lines, running in ~2 seconds. The
suite is the other half of understanding the code: it shows which behaviors
are treated as promises and which as incidental.

### Layout

| File | Lines | What it covers |
|---|---|---|
| `conftest.py` | 41 | Shared fixtures: `repo`, `nb_file`, `make_notebook` helper |
| `test_notebook.py` | 187 | Cell read/write: count, cat, edit, append, insert, remove, tag |
| `test_inspect.py` | 142 | list_ipynb, repo_map, resolve_ipynb, find_in_src |
| `test_scaffold.py` | 161 | Notes, articles, courses, chapters, sections, sidebar registration |
| `test_convert.py` | 112 | import_notebook (flat tiers), import_chapter (courses) |
| `test_cli.py` | 30 | Only the UTF-8 stdin/stdout helpers |
| `test_paths.py` | 8 | Smoke test: `repo_root()` finds `AGENTS.md` + `pyproject.toml` |
| `fixtures/quarto.yml` | 3 | Minimal `_quarto.yml` copied into each isolated repo |

### The isolation pattern

Every filesystem-touching test uses the same shape: `tmp_path` +
`monkeypatch.chdir(tmp_path)`, so the repo's real `notes/` and
`_quarto.yml` are never touched. Two fixtures do the heavy lifting:

```python
@pytest.fixture
def repo(tmp_path, monkeypatch):
    """Isolated repo root: cwd = tmp_path with a minimal _quarto.yml."""
    monkeypatch.chdir(tmp_path)
    shutil.copy(FIXTURES_DIR / "quarto.yml", tmp_path / "_quarto.yml")
    return tmp_path

@pytest.fixture
def nb_file(tmp_path, monkeypatch):
    """A 3-cell notebook at notes/test.ipynb with cwd = tmp_path."""
    monkeypatch.chdir(tmp_path)
    return make_notebook(
        tmp_path / "notes" / "test.ipynb",
        [nbformat.v4.new_markdown_cell("# Title"),
         nbformat.v4.new_code_cell("print('hello')"),
         nbformat.v4.new_markdown_cell("## Section")],
    )
```

`repo` is for anything that needs `_quarto.yml` (scaffold, convert);
`nb_file` is for cell operations that just need a notebook on disk. Both
rely on `paths.py` resolving relative to cwd, which is why the tests
`chdir` into the temp dir rather than passing absolute paths around.

### What the tests pin down

The suite tests through the **public API** — calling `notebook.edit_cell`,
`scaffold.new_course`, and so on — not internal helpers. A few tests reach
for a private function (`scaffold._load_yaml`,
`scaffold._find_course_sidebar_entry`) when the only way to verify a side
effect is to read the YAML back, but those are exceptions.

The interesting tests check invariants, not just the happy path:

- **`test_edit_cell_preserves_other_cells`**: edits cell 0, asserts cell 1
  is untouched. The "outputs survive" guarantee in miniature — if
  `edit_cell` ever rewrote the whole notebook, this test would catch it.
- **`test_remove_cell_by_tag_removes_all`**: tags two cells, removes by
  tag, asserts one cell remains. Pins the reverse-sorted deletion behavior
  (delete from highest index first so earlier indices stay valid).
- **`test_tag_cell_read_only_does_not_write`**: captures `mtime` before
  and after a read-only `tag_cell` call, asserts equality. A
  filesystem-touching bug in the read path fails this with no assertion
  about content.
- **`test_new_course_idempotent_registration`**: calls `new_course`
  twice, asserts the sidebar has exactly one entry with that id. Guards
  the idempotency claim in `_register_course`.
- **`test_find_is_case_insensitive`** and **`test_find_searches_courses`**:
  the two non-obvious behaviors of `find_in_src` (lowercasing the query,
  recursing into nested course dirs) each get a dedicated test.
- **`test_cat_range`**: `--index "0:2"` returns cells 0 and 1, not 2.
  Pins the half-open Python-slice semantics of `_parse_index_spec`.

### What is deliberately not tested

Three modules have no tests at all, a conscious trade-off rather than an
oversight:

- **`render.py`** shells out to `quarto` — testing it means installing
  Quarto in CI and rendering real notebooks. The cost outweighs the value
  for a 60-line subprocess wrapper.
- **`resume.py`** shells out to `pdflatex` and depends on Jinja2
  templates + a YAML file. Same reasoning. The reproducible-build logic
  (`SOURCE_DATE_EPOCH`, stable scratch dir) is the interesting part, but
  it is only observable by diffing PDF bytes, which is brittle.
- **`vault.py`** talks to the OS keyring. Testing it means mocking
  `keyring` or hitting the real keychain (which stores real secrets).
  Neither is worth it for 43 lines of straightforward wrapper code.

**`test_cli.py`** tests only the two UTF-8 helpers (`_read_stdin`,
`_force_utf8_streams`), not full command invocation. The reason is
architectural: every CLI command is a thin wrapper around a domain
function already tested directly. Testing through Typer's runner would
duplicate the domain tests and add Typer/Click version coupling for little
gain. The one thing worth testing at the CLI layer — that Unicode
survives the stdin → notebook path on any platform — gets a focused test
with a `_FakeStdin` stand-in.

### What the test map says about the code

The suite is a map of where the code is load-bearing. The notebook core
(`notebook.py`, `inspect.py`) carries the most tests because that is where
data loss lives: a bad `edit_cell` clobbers a notebook, a broken
`resolve_ipynb` breaks every command. Scaffold and convert are tested
because they write to the filesystem and mutate `_quarto.yml`, and a
sidebar bug is silent until a missing chapter appears on the site. The
peripheral subsystems are untested because they are swappable wrappers
around external tools, and the cost of testing them is disproportionate
to the risk.

Reading the source alongside the tests for each module is the fastest way
to a mental model: the tests mark which behaviors are promises and which
are incidental.


## Core vs. peripheral {#core-vs-peripheral}

- **Core notebook layer:** `notebook.py` (cell read/render/mutate),
  `inspect.py` (resolution + search + map), `paths.py`, `scaffold.py` +
  `convert.py` (creation/import). All traffic in `.ipynb` files.
- **Peripheral:** `vault.py` (keyring secrets, completely disjoint),
  `resume.py` (independent YAML→PDF/home-page pipeline), `render.py`
  (Quarto subprocess wrappers). Swappable without touching the notebook
  layer.


## Design patterns worth knowing {#design-patterns-worth-knowing}

1. **Stateless, functional design.** No central `Notebook` class; every
   operation does `resolve_ipynb(name) → nbformat.read → mutate →
   nbformat.write` fresh. Safe for a CLI: no stale in-memory state across
   commands.
2. **Central name resolution funnel.** Every notebook op calls
   `inspect.resolve_ipynb`, implementing the bare-stem / tier-prefixed /
   full-path ladder. One choke point, one way to find a notebook.
3. **One error pattern everywhere.** Domain functions raise stdlib
   exceptions with actionable messages; `cli.py` catches them centrally in
   `_user_error()` → red message + `typer.Exit(1)`. Modules never print or
   exit; the CLI never knows domain details.
4. **Mutation safeguards.** Index-only locators on writes (tags/labels are
   read-only locators); `_resolve_unique_cell` rejects zero/multiple
   matches; `edit_cell` touches only the source so inline outputs survive;
   20k char cap; `remove_cell` deletes reverse-sorted so earlier indices
   stay valid.
5. **Built for LLM consumption.** `cat` defaults to 4096-char slices with
   `src[start:end] of total` headers for chainable reads without re-paying
   bytes; non-text outputs summarized, not dumped.
6. **Deterministic builds (resume).** `SOURCE_DATE_EPOCH` + stable scratch
   dir → byte-identical re-renders.
7. **YAML round-trip preservation (scaffold).** ruamel `typ="rt"` so
   sidebar edits never clobber user comments/formatting; idempotent
   `_register_course`.
8. **Perf pragmatism.** ripgrep pre-filter for `find` before JSON-parse
   only candidates; graceful `shutil.which("rg")` fallback.
9. **UTF-8 hardening.** Streams reconfigured + stdin read from `.buffer`
   at startup: one deterministic encoding for cell writes on any platform.


## Errors & constraints cheat-sheet {#errors-cheat-sheet}

- `wt` commands must run via the venv: `.venv/bin/wt` (never bare `wt`).
- Cell writes capped at 20k chars; break large content into smaller cells.
- Mutations take numeric `--index` only; tags/labels locate via `wt cat`
  first, then pass the reported index.
- Indices shift after insert/remove: plan mutations right-to-left, and
  re-resolve indices after any insert/remove.
- Secrets: OS keyring only (`.watchtower/vault_keys.json` is the gitignored
  key index). Never commit secret values.
- Doc-drift: changes to the `wt` CLI surface must be reflected in both
  `AGENTS.md` and `README.md`.
